# 📓 Audio AI Module 3: HuBERT & Interactive Audacity-Style Phoneme Discovery
Welcome to Module 3! In this notebook, we explore **HuBERT (Hidden-Unit BERT)** by Hsu et al. (Meta, 2021).

---

## 💡 Key Concepts & Notebook Features

### 1. Independence of $K$ (Clusters) and Temporal Segmentation
* **$N$ (Audio Frames):** Fixed by audio length and model stride ($1$ frame every $20\text{ ms} = 50\text{ Hz}$).
* **$K$ (Cluster Vocabulary):** The number of unique acoustic centroids.
* **Contiguous Span Merging:** In speech, adjacent $20\text{ ms}$ frames usually share the same cluster ID (e.g., a held vowel `[3, 3, 3, 3, 3]`). By merging contiguous identical IDs into unified spans, $5$ frames collapse into **$1$ phonetic segment** lasting $100\text{ ms}$!

### 2. Audacity-Style Interactive Playback Scrubbing
This notebook embeds a custom HTML5 + JavaScript player inside `ipywidgets`:
* A **red vertical cursor line** moves across the waveform and token trajectory in real time as audio plays.
* **Click-to-Seek:** Clicking anywhere on the plot instantly jumps the audio playback to that exact timestamp!

In [ ]:
import os
import io
import base64
import urllib.request
import torch
import torchaudio
import torchaudio.transforms as T
import numpy as np
import scipy.io.wavfile as wavfile
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets
from ipywidgets import interact, Dropdown, IntSlider

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Download & Load Real Speech Sample
We load a clean $16\text{ kHz}$ human speech audio utterance.

In [ ]:
# Load PyTorch official speech sample asset
audio_filename = "speech_sample.wav"
if not os.path.exists(audio_filename):
    # Hello
    # url = "https://storage.googleapis.com/cloud-samples-data/speech/hello.wav"
    # Longer sentence
    url = "https://cdn-media.huggingface.co/speech_samples/sample1.flac"
    urllib.request.urlretrieve(url, audio_filename)

    if url.endswith(".flac"):
        temp_waveform, temp_sr = torchaudio.load(url)
        torchaudio.save(audio_filename, temp_waveform, temp_sr)

# Load audio using torchaudio
waveform, sample_rate = torchaudio.load(audio_filename)

# Ensure 16 kHz for HuBERT [1, T]
target_sr = 16000
if sample_rate != target_sr:
    waveform = T.Resample(orig_freq=sample_rate, new_freq=target_sr)(waveform)
    sample_rate = target_sr

# Ensure mono audio
if waveform.shape[0] > 1:
    waveform = torch.mean(waveform, dim=0, keepdim=True)

print(f"Real Speech Utterance Loaded!")
print(f"Shape: {waveform.shape}, Sample Rate: {sample_rate} Hz, Duration: {waveform.shape[1]/sample_rate:.2f}s")

## 2. Load Pre-trained HuBERT Model & Extract Layer Activations
We load Meta's pre-trained **HuBERT Base** model (`facebook/hubert-base-ls960` or `torchaudio.pipelines.HUBERT_BASE`).
We pass our audio waveform through the model and extract frame-level hidden state activations across Transformer layers.

In [ ]:
# Load HuBERT Model
try:
    from transformers import HubertModel
    hubert_model = HubertModel.from_pretrained("facebook/hubert-base-ls960").to(device)
    hubert_model.eval()
    backend = "transformers"
    print("Loaded HuBERT Base via Hugging Face transformers")
except Exception:
    bundle = torchaudio.pipelines.HUBERT_BASE
    hubert_model = bundle.get_model().to(device)
    hubert_model.eval()
    backend = "torchaudio"
    print("Loaded HuBERT Base via torchaudio.pipelines")

@torch.no_grad()
def extract_layer_activations(wav_tensor):
    wav_input = wav_tensor.to(device)
    activations = {}
    
    if backend == "transformers":
        outputs = hubert_model(wav_input, output_hidden_states=True)
        hidden_states = outputs.hidden_states
        activations['Conv Output'] = hidden_states[0].squeeze(0).cpu().numpy()
        activations['Layer 1'] = hidden_states[1].squeeze(0).cpu().numpy()
        activations['Layer 6'] = hidden_states[6].squeeze(0).cpu().numpy()
        activations['Layer 12'] = hidden_states[12].squeeze(0).cpu().numpy()
    else:
        feats, hidden_states = hubert_model.extract_features(wav_input)
        activations['Conv Output'] = feats.squeeze(0).cpu().numpy()
        activations['Layer 1'] = hidden_states[0].squeeze(0).cpu().numpy()
        activations['Layer 6'] = hidden_states[5].squeeze(0).cpu().numpy()
        activations['Layer 12'] = hidden_states[11].squeeze(0).cpu().numpy()
        
    return activations

layer_activations = extract_layer_activations(waveform)
frame_count = layer_activations['Layer 6'].shape[0]
print(f"Extracted activations across layers! Number of frames: {frame_count} (1 frame/20ms)")

## 3. K-Means Clustering & Contiguous Span Merging Helper
We cluster continuous $20\text{ ms}$ vectors into $K$ centroids, then merge contiguous identical cluster IDs to extract clean phonetic segment boundaries.

In [ ]:
def perform_kmeans_clustering(features, n_clusters=15):
    kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
    cluster_labels = kmeans.fit_predict(features)
    return cluster_labels

def merge_contiguous_spans(cluster_labels):
    """
    Merges adjacent identical frame cluster IDs into unified phonetic spans
    Returns list of (start_frame, end_frame, cluster_id)
    """
    merged_spans = []
    curr_id = cluster_labels[0]
    start_i = 0
    for i, cid in enumerate(cluster_labels):
        if cid != curr_id:
            merged_spans.append((start_i, i, curr_id))
            curr_id = cid
            start_i = i
    merged_spans.append((start_i, len(cluster_labels), curr_id))
    return merged_spans

## 🎛️ 4. Interactive Audacity-Style Visualizer (Aligned $t=0$)
Adjust the **HuBERT Layer** dropdown and **Clusters ($K$)** slider below.

The graph x-axis is explicitly anchored at $t=0.0$, and fixed canvas margins ensure $100\%$ pixel alignment between the red Audacity playback cursor and the speech waveform!

In [ ]:
def visualize_audacity_phonemes(selected_layer='Layer 6', num_clusters=15):
    feats = layer_activations[selected_layer]
    cluster_labels = perform_kmeans_clustering(feats, n_clusters=num_clusters)
    merged_spans = merge_contiguous_spans(cluster_labels)
    
    wav_np = waveform.squeeze().cpu().numpy()
    duration = len(wav_np) / sample_rate
    num_frames = len(cluster_labels)
    dt = duration / num_frames
    
    # Render Matplotlib Figure with fixed margins for exact JS cursor alignment
    fig, axes = plt.subplots(2, 1, figsize=(12, 5.5), sharex=True)
    fig.subplots_adjust(left=0.08, right=0.98, top=0.92, bottom=0.10, hspace=0.25)
    
    time_samples = np.linspace(0, duration, len(wav_np))
    time_frames = np.linspace(0, duration, num_frames)
    
    # 1. Waveform with Merged Phonetic Spans (Strictly xlim [0, duration])
    axes[0].plot(time_samples, wav_np, color='black', alpha=0.6, linewidth=1)
    cmap = plt.cm.get_cmap('tab20', num_clusters)
    for start_i, end_i, cid in merged_spans:
        axes[0].axvspan(start_i * dt, end_i * dt, color=cmap(cid % 20), alpha=0.35)
        
    axes[0].set_xlim(0, duration)
    axes[0].set_title(f"Audacity View: HuBERT ({selected_layer}) Pseudo-Phonemes (K={num_clusters}) — {len(merged_spans)} Discovered Spans")
    axes[0].set_ylabel("Amplitude")
    axes[0].grid(True, linestyle='--', alpha=0.3)
    
    # 2. Token Trajectory Plot (Strictly xlim [0, duration])
    axes[1].step(time_frames, cluster_labels, where='post', color='#1f77b4', linewidth=2)
    axes[1].set_xlim(0, duration)
    axes[1].set_xlabel("Time (seconds)")
    axes[1].set_ylabel("Cluster ID")
    axes[1].set_yticks(range(0, num_clusters, max(1, num_clusters // 10)))
    axes[1].grid(True, linestyle='--', alpha=0.5)
    
    # Save Matplotlib Figure to Base64 PNG without tight_layout / bbox_inches='tight'
    # This guarantees left=0.08 and right=0.98 padding remain perfectly fixed!
    buf_img = io.BytesIO()
    plt.savefig(buf_img, format='png', dpi=120)
    plt.close(fig)
    img_b64 = base64.b64encode(buf_img.getvalue()).decode('utf-8')
    
    # Convert Audio Tensor to Base64 WAV
    wav_int16 = (wav_np / np.max(np.abs(wav_np)) * 32767).astype(np.int16)
    buf_wav = io.BytesIO()
    wavfile.write(buf_wav, sample_rate, wav_int16)
    audio_b64 = base64.b64encode(buf_wav.getvalue()).decode('utf-8')
    
    uid = f"audacity_{np.random.randint(100000, 999999)}"
    
    html_code = f"""
    <div style="width: 100%; max-width: 900px; font-family: sans-serif;">
        <div id="plot-container-{uid}" style="position: relative; cursor: pointer; display: inline-block; width: 100%;">
            <img id="plot-img-{uid}" src="data:image/png;base64,{img_b64}" style="width: 100%; display: block;" />
            <div id="scrub-line-{uid}" style="position: absolute; top: 0; bottom: 0; left: 8%; width: 2px; background-color: red; pointer-events: none; display: none;"></div>
        </div>
        <div style="margin-top: 8px;">
            <audio id="audio-{uid}" controls style="width: 100%;">
                <source src="data:audio/wav;base64,{audio_b64}" type="audio/wav">
            </audio>
        </div>
    </div>

    <script>
    (function() {{
        const audio = document.getElementById('audio-{uid}');
        const scrubLine = document.getElementById('scrub-line-{uid}');
        const container = document.getElementById('plot-container-{uid}');
        
        // Exact Matplotlib subplots_adjust bounds (left=0.08, right=0.98)
        const leftMarginPct = 0.08; 
        const rightMarginPct = 0.98;
        const plotWidthPct = rightMarginPct - leftMarginPct;

        audio.ontimeupdate = function() {{
            if (audio.duration) {{
                scrubLine.style.display = 'block';
                const progress = audio.currentTime / audio.duration;
                const linePos = (leftMarginPct + (progress * plotWidthPct)) * 100;
                scrubLine.style.left = linePos + '%';
            }}
        }};

        container.onclick = function(e) {{
            const rect = container.getBoundingClientRect();
            const clickX = (e.clientX - rect.left) / rect.width;
            
            if (clickX >= leftMarginPct && clickX <= rightMarginPct) {{
                const normalizedX = (clickX - leftMarginPct) / plotWidthPct;
                audio.currentTime = normalizedX * audio.duration;
                audio.play();
            }}
        }};
    }})();
    </script>
    """
    
    clear_output(wait=True)
    print(f"Layer Selected: {selected_layer} | Vocabulary Size (K): {num_clusters}")
    print(f"Total Frames: {num_frames} (1 frame/20ms) | Discovered Phonetic Spans: {len(merged_spans)}")
    display(HTML(html_code))

# Interactive Widgets
interact(visualize_audacity_phonemes,
         selected_layer=Dropdown(
             options=['Conv Output', 'Layer 1', 'Layer 6', 'Layer 12'],
             value='Layer 6',
             description='HuBERT Layer:'
         ),
         num_clusters=IntSlider(
             value=12,
             min=4,
             max=24,
             step=4,
             description='Clusters (K):'
         ));

If you change from Conv Output to Layer 6 you will see changes. Indeed, the cluster assignments flicker rapidly because early layers capture high-frequency pitch and raw acoustic noise


The higher layer you go, the smoother the cluster assignment should become. 